# Explainability and Trust
### Part 3 — Explaining a model you cannot check with the methods from part 2

**Author**: Jesper Dramsch (ECMWF)

*This notebook was last tested and operational on 01/10/2026. Please [report any issues](https://github.com/ecmwf-training/2026-ml-esm-training/issues).*

<!-- :::{admonition} About
:class: note, dropdown -->
This notebook was adapted for the DestinE [2026 Machine Learning for Earth System Modelling Course (Applications and Future Directions)](https://learning.ecmwf.int/enrol/index.php?id=100), where it accompanies the Part 3 video of Module 4, *Explainability & Trust*. It is adapted from the [xai-weather-workshop](https://github.com/JesperDramsch/xai-weather-workshop) (CLIMADEMIC Summer School, July 2026, MIT licence), which adapted it from ECMWF's 2025 ML training course (`ecmwf-training/2025-ml-training`, notebook `5-xAI-with-AIFS/explain_AIFS-cpu.ipynb`). See the workshop's [NOTICE](https://github.com/JesperDramsch/xai-weather-workshop/blob/main/NOTICE) for full attribution.
<!-- ::: -->

<!-- :::{admonition} Running this notebook
:class: tip, dropdown -->
You may try to run/access this notebook on the free online platforms linked below. Please note they are not officially supported by or linked with ECMWF.

[![colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ecmwf-training/2026-ml-esm-training/blob/main/c3/m4/AIFS_sensitivities.ipynb)
[![kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/ecmwf-training/2026-ml-esm-training/blob/main/c3/m4/AIFS_sensitivities.ipynb)
[![github](https://img.shields.io/badge/Open%20in-GitHub-black?logo=github)](https://github.com/ecmwf-training/2026-ml-esm-training/blob/main/c3/m4/AIFS_sensitivities.ipynb)

The notebook is self-contained and runs on a CPU. It uses a pre-computed array of AIFS output (`artifacts/sensitivities_o48.npz`) and two images (in `artifacts/images/`). If these files are not next to the notebook (e.g. on Colab or Kaggle), the notebook downloads them from the course repository. On Kaggle, remember to switch on internet access in the notebook settings.
<!--
::: -->


## Introduction

In Part 2 (`Classic_xai.ipynb`), we explained a Random Forest and an SVM classifier that predict penguin species. Most of the methods we used there probe the model from the outside: they change the inputs and look at how the output changes. This is how feature permutation importance, PCP/ICE curves and Shapley analysis works.

In this notebook, we attempt to explain ECMWF's **AIFS** (Artificial Intelligence Forecasting System), an operational weather model built on a graph neural network. AIFS has millions of input values (every variable at every grid point and pressure level), so probing it in the same way is not possible: It takes far too long. Instead, we can use a *backward sensitivity*, also called an *adjoint sensitivity*. It uses the gradients of the network to answer *"if each input value were changed slightly, how much would one particular output of the forecast change?"*

This notebook is a demonstration with no exercises, but with some interactive parts. We use a pre-computed array of real AIFS output sensitivities (i.e. *gradients* of the AIFS network), produced with the sensitivity code from ECMWF's 2025 ML training course.

---
## Prepare your environment

This notebook uses a small set of packages:
- `numpy` for loading and handling the sensitivity array
- `matplotlib` and `seaborn` for static plots and animations
- `plotly` for the interactive network diagrams and map
- `cmocean` for the diverging colour map

Run the cell below to install them. This may take a minute.

In [ ]:
%pip install -q -r https://raw.githubusercontent.com/ecmwf-training/2026-ml-esm-training/main/c3/m4/requirements.txt

### Load the sensitivities

The next cell loads the pre-computed sensitivity array (and downloads it first, if needed). It also defines two helper functions: one to display the pre-rendered images, and one to compute distances on the globe.

In [ ]:
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


ARTIFACTS_DIR = Path("artifacts")
PERTURB_LOCATION = (40.0, 120.0)  # (lat, lon) -- the real example from the source notebook
LEVELS = [50, 100, 150, 200, 250, 300, 400, 500, 600, 700, 850, 925, 1000]
TIMESTEP_LABELS = ["-6H", "0H"]

# Colab and Kaggle only fetch this single notebook file, not the rest of the repository --
# download the committed data/images it needs from the course repository if they are missing.
ARTIFACTS_URL = "https://raw.githubusercontent.com/ecmwf-training/2026-ml-esm-training/main/c3/m4/artifacts/"
for _f in ["sensitivities_o48.npz", "images/01_pressure_level_summary.png", "images/05_cross_section_temperature.png"]:
    _dest = ARTIFACTS_DIR / _f
    if not _dest.exists():
        _dest.parent.mkdir(parents=True, exist_ok=True)
        print(f"Downloading {_f} ...")
        urllib.request.urlretrieve(ARTIFACTS_URL + _f, _dest)

sensitivities = np.load(ARTIFACTS_DIR / "sensitivities_o48.npz", allow_pickle=False)
lat = sensitivities["latitudes"]
lon = sensitivities["longitudes"]
lon_wrapped = np.where(lon > 180, lon - 360, lon)


def show_artifact(filename, title=None, figsize=(11, 6)):
    # Display a pre-rendered artifact image with a clean, axis-free figure.
    img = plt.imread(ARTIFACTS_DIR / "images" / filename)
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(img)
    ax.axis("off")
    if title:
        ax.set_title(title, fontsize=12)
    plt.tight_layout()
    plt.show()


def haversine_km(lat1, lon1, lat2, lon2):
    # Great-circle distance in km -- same formula the source runner uses to pick perturbation-radius grid points.
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlambda / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

---
## The problem we'll study

AIFS has produced a 6-hour forecast. We pick one output of that forecast: the 2 m temperature (2t) at all grid points within 350 km of 40°N, 120°E, on the North China coast near the Bohai Sea (close to Beijing and Tianjin). We then ask: *how much would this forecast 2t change if we changed each value of the initial atmospheric state slightly?*

The answer is one number per input value: its **sensitivity**. A large positive sensitivity means that increasing that input would raise the forecast 2t near 40°N, 120°E. A large negative sensitivity means it would lower it. A sensitivity close to zero means the forecast 2t there hardly depends on that input. AIFS takes the state at two times as input, 6 hours before the forecast start (−6H) and at the forecast start (0H), so we get sensitivities for both.

Mathematically, the sensitivity is the gradient of the selected output with respect to every input. It is often written as the vector-Jacobian product $v^\top \partial y / \partial x$, where $y$ is the forecast, $x$ the initial state, and $v$ selects the output we care about: $v$ is 1 for 2t within the 350 km circle, and 0 everywhere else. The PyTorch automatic differentiation function (`torch.autograd.functional.vjp`) that computes these sensitivities is just the same tool used to train neural networks with backpropagation.

**A note on normalisation before you evaluate the sensitivities**
These sensitivities were computed in the model's own *normalised* units: before AIFS receives its input features, each variable is rescaled, usually by subtracting its mean and dividing by its standard deviation. So a sensitivity of 0.9 for `z_850` and of 0.4 for `2t` both mean "change in forecast 2t *per standard deviation of that input*", and so it makes sense to compare them. In physical units (m² s⁻² for geopotential, K for temperature), the numbers would be different, and so would any ranking of which input matters most. In the code below, the normalisation is handled by the settings `perturb_normalised_space=True` and `return_unnormalised_sensitivities=False`.

## How is a backward sensitivity computed?

The diagram below shows the idea. The top row is an ordinary AIFS forecast: the initial state goes in, and the forecast comes out. The bottom row goes the other way through the same network: we start from the output we selected ($v$) and work backwards, to find how sensitive that output is to every input.

In [ ]:
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
import seaborn as sns

palette = sns.color_palette("colorblind")
FORWARD_COLOR, BACKWARD_COLOR = palette[0], palette[1]

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
ax.axis("off")


def _box(x, y, w, h, text, color, fontsize=10.5):
    patch = FancyBboxPatch(
        (x, y), w, h, boxstyle="round,pad=0.08,rounding_size=0.12",
        linewidth=1.8, edgecolor=color, facecolor=color, alpha=0.18,
    )
    ax.add_patch(patch)
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=fontsize, color="black")


def _arrow(x1, y1, x2, y2, color, lw=2.2, ls="solid", label=None):
    arr = FancyArrowPatch((x1, y1), (x2, y2), arrowstyle="-|>", mutation_scale=18, linewidth=lw, color=color, linestyle=ls)
    ax.add_patch(arr)
    if label:
        ax.text((x1 + x2) / 2, max(y1, y2) + 0.28, label, ha="center", va="bottom", fontsize=9.5, color=color)


_box(0.3, 3.6, 2.5, 1.4, "Initial state $x$\n(atmosphere,\n-6h and 0h)", FORWARD_COLOR)
_box(3.6, 3.6, 2.6, 1.4, "AIFS forward pass\n$y = f(x)$", FORWARD_COLOR)
_box(7.0, 3.6, 2.6, 1.4, "Forecast $y$\n(+6h, all variables)", FORWARD_COLOR)
_arrow(2.8, 4.3, 3.6, 4.3, FORWARD_COLOR, label="forward")
_arrow(6.2, 4.3, 7.0, 4.3, FORWARD_COLOR)
ax.text(5.15, 5.35, "FORWARD PASS -- an ordinary AIFS forecast", ha="center", fontsize=11, color=FORWARD_COLOR, weight="bold")

_box(7.0, 0.6, 2.6, 1.4, "From the forecast $y$, \nover the domain set by $v$ \n(2t near 40N, 120E)", BACKWARD_COLOR)
_box(3.6, 0.6, 2.6, 1.4, "Reverse-mode autodiff\n$v^\\top \\dfrac{\\partial y}{\\partial x}$\n(torch.autograd\n.functional.vjp)", BACKWARD_COLOR, fontsize=9.5)
_box(0.3, 0.6, 2.5, 1.4, "Sensitivity map to inputs \n(figures in this notebook)", BACKWARD_COLOR)
_arrow(7.0, 1.3, 6.2, 1.3, BACKWARD_COLOR, label="backward")
_arrow(3.6, 1.3, 2.8, 1.3, BACKWARD_COLOR)
ax.text(5.15, 0.25, "BACKWARD PASS -- differentiate through the network to get sensitivity of output to input", ha="center", fontsize=11, color=BACKWARD_COLOR, weight="bold")

_arrow(8.3, 3.6, 8.3, 2.0, "grey", lw=1.4, ls="dashed")
ax.text(8.55, 2.8, "same $f$,\nnow\nin reverse", ha="left", va="center", fontsize=8.5, color="grey")

fig.suptitle("How a backward (adjoint) sensitivity is computed", fontsize=13, y=0.99)
plt.tight_layout(rect=[0, 0.04, 1, 0.96])
plt.show()

### Try it yourself: forward, then backward

AIFS has far too many parameters to follow by hand, so let us first look at a small network example instead: It has 2 inputs (`x1` and `x2`), connected to 4 hidden units with a `tanh` activation, which are summed into 1 output (`y`). It is small enough that we can compute its gradients by hand with the chain rule. We use it in two plots:

1. **Forward pass:** move `x1` or `x2`, and see how the hidden units and the output `y` respond.
2. **Backward pass:** keep the inputs fixed, choose how much we care about the output (`v`), and see how that traces back to the inputs.

In both plots, the node colours show the values of the inputs, hidden units and output. The edges from the inputs to the hidden units show each path's contribution to the gradient: red is positive, blue is negative, and wider edges contribute more. The grey edges show the fixed weights from the hidden units to the output. The bar chart shows the total sensitivity of the output to each input.

In [ ]:
import cmocean
import plotly.graph_objects as go
from IPython.display import HTML

# A real feedforward net: 2 inputs, fully connected to 4 tanh hidden units, summed into 1 output.
# Hand-differentiated -- no autograd library needed, small enough for the chain rule by hand.
toy_W1 = np.array([2.0, 1.0, -1.5, 0.8])    # x1 -> hidden weights
toy_W2 = np.array([0.5, -1.8, 1.0, 1.3])    # x2 -> hidden weights
toy_B = np.array([0.0, 1.0, -0.5, -1.5])    # hidden biases
toy_V = np.array([1.2, -0.8, 1.5, 0.6])     # hidden -> output weights
toy_C = 0.0
N_TOY = len(toy_W1)

toy_input_pos = [(0, 1.0), (0, -1.0)]
toy_hidden_pos = [(1, 2.1 - i * 1.4) for i in range(N_TOY)]
toy_output_pos = (2, 0)
TOY_BALANCE = cmocean.cm.balance


def _toy_to_rgb(cmap, val, vmin, vmax):
    t = np.clip((val - vmin) / (vmax - vmin + 1e-12), 0, 1)
    r, g, b, _a = cmap(t)
    return f"rgb({int(r*255)},{int(g*255)},{int(b*255)})"


def _toy_backward(x1, x2, v):
    h = np.tanh(toy_W1 * x1 + toy_W2 * x2 + toy_B)
    y = float(np.dot(toy_V, h) + toy_C)
    dh = 1 - h**2
    edge1 = v * toy_V * dh * toy_W1  # each x1->hidden_i->output path's contribution to v*dy/dx1
    edge2 = v * toy_V * dh * toy_W2  # each x2->hidden_i->output path's contribution to v*dy/dx2
    return h, y, edge1, edge2, float(edge1.sum()), float(edge2.sum())


def _toy_frame(x1, x2, v, name, title_fn, show_halo=False, bar_labels=("dy/dx1", "dy/dx2")):
    h, y, edge1, edge2, dydx1, dydx2 = _toy_backward(x1, x2, v)
    max_edge = 3.0  # fixed reference so widths are comparable across every frame in both plots

    in_colors = [_toy_to_rgb(TOY_BALANCE, x1, -3, 3), _toy_to_rgb(TOY_BALANCE, x2, -3, 3)]
    in_text = [f"x1={x1:.2f}", f"x2={x2:.2f}"]
    hid_colors = [_toy_to_rgb(TOY_BALANCE, hv, -1, 1) for hv in h]
    hid_text = [f"h{i+1}={h[i]:.2f}" for i in range(N_TOY)]

    edges = []
    for i in range(N_TOY):
        w = 0.5 + 7 * abs(edge1[i]) / max_edge
        color = "crimson" if edge1[i] > 0 else "steelblue"
        edges.append(go.Scatter(x=[toy_input_pos[0][0], toy_hidden_pos[i][0]], y=[toy_input_pos[0][1], toy_hidden_pos[i][1]],
                                 mode="lines", line=dict(width=w, color=color), showlegend=False, hoverinfo="skip"))
    for i in range(N_TOY):
        w = 0.5 + 7 * abs(edge2[i]) / max_edge
        color = "crimson" if edge2[i] > 0 else "steelblue"
        edges.append(go.Scatter(x=[toy_input_pos[1][0], toy_hidden_pos[i][0]], y=[toy_input_pos[1][1], toy_hidden_pos[i][1]],
                                 mode="lines", line=dict(width=w, color=color, dash="dash"), showlegend=False, hoverinfo="skip"))
    for i in range(N_TOY):
        w = 0.5 + 3 * abs(toy_V[i]) / np.abs(toy_V).max()
        edges.append(go.Scatter(x=[toy_hidden_pos[i][0], toy_output_pos[0]], y=[toy_hidden_pos[i][1], toy_output_pos[1]],
                                 mode="lines", line=dict(width=w, color="lightgrey"), showlegend=False, hoverinfo="skip"))

    # The halo around the output node is the only visual representation of v -- v has no node
    # of its own and never touches the real forward-pass values (x1, x2, h, y). Without it, the
    # diagram would look frozen while v moves, since nothing else in the plot represents v itself.
    halo_color = "rgba(220,20,60,0.30)" if v >= 0 else "rgba(70,130,180,0.30)"
    halo_trace = go.Scatter(x=[toy_output_pos[0]], y=[toy_output_pos[1]], mode="markers",
                             marker=dict(size=36 + 22 * abs(v), color=halo_color, line=dict(width=0)),
                             showlegend=False, hoverinfo="skip", visible=show_halo)

    in_trace = go.Scatter(x=[p[0] for p in toy_input_pos], y=[p[1] for p in toy_input_pos], mode="markers+text",
                           text=in_text, textposition="middle left",
                           marker=dict(size=36, color=in_colors, line=dict(width=2, color="black")), showlegend=False)
    hid_trace = go.Scatter(x=[p[0] for p in toy_hidden_pos], y=[p[1] for p in toy_hidden_pos], mode="markers+text",
                            text=hid_text, textposition="middle right",
                            marker=dict(size=34, color=hid_colors, line=dict(width=2, color="black")), showlegend=False)
    out_trace = go.Scatter(x=[toy_output_pos[0]], y=[toy_output_pos[1]], mode="markers+text", text=[f"y={y:.2f}"],
                            textposition="top center",
                            marker=dict(size=36, color=_toy_to_rgb(TOY_BALANCE, y, -3, 2), line=dict(width=2, color="black")),
                            showlegend=False)
    bar_trace = go.Bar(x=list(bar_labels), y=[dydx1, dydx2], marker=dict(color=["#4C72B0", "#DD8452"]),
                        showlegend=False, xaxis="x2", yaxis="y2")

    return go.Frame(data=edges + [halo_trace, in_trace, hid_trace, out_trace, bar_trace], name=name,
                     layout=go.Layout(annotations=[dict(text=title_fn(x1, x2, v, y, dydx1, dydx2), x=0.28, y=1.14,
                                                          xref="paper", yref="paper", showarrow=False, font=dict(size=12))]))


def _toy_base_layout():
    return dict(
        xaxis=dict(domain=[0, 0.55], visible=False, range=[-0.8, 2.4]),
        yaxis=dict(domain=[0, 1], visible=False, range=[-2.6, 2.6]),
        xaxis2=dict(domain=[0.68, 1], anchor="y2"),
        yaxis2=dict(title="sensitivity", range=[-6, 6], zeroline=True, anchor="x2", side="left"),
        width=1050,
    )

#### Plot 1: forward pass

Each input has its own slider; moving one keeps the other at 0, so you can see each input's effect separately. As you move a slider, the output `y` changes: this is an ordinary forward pass. Additionally, the bar chart shows the *gradients* `dy/dx1` and `dy/dx2` at the current point, i.e. the sensitivities that the backward pass in Plot 2 computes more explicitly.

Notice what happens when you push `x1` or `x2` beyond about ±2: most hidden units get close to +1 or −1. Remember that edge thickness quantifies how much that edge contributes to the gradient $dy/dx$ for that edge's output ($y$) and input ($x$). This is because `y=tanh(x)` becomes flat in `x` beyond ±2, so most gradients vanish to zero. This means that same network can therefore be very sensitive to an input in one situation, and hardly sensitive at all in another. That is, a backward sensitivity only describes the model at the point where it is computed.

In [ ]:
toy_title_fwd = lambda x1, x2, v, y, d1, d2: (
    f"FORWARD PASS -- x1={x1:.2f}  x2={x2:.2f}  =>  y={y:.2f}"
    f"<br>(sensitivities at this point, for later: dy/dx1={d1:.2f}, dy/dx2={d2:.2f})"
)

TOY_X1_RANGE = np.linspace(-3, 3, 41)
TOY_X2_RANGE = np.linspace(-3, 3, 41)
toy_x1_frames = [_toy_frame(x1, 0.0, 1.0, f"x1_{i}", toy_title_fwd) for i, x1 in enumerate(TOY_X1_RANGE)]
toy_x2_frames = [_toy_frame(0.0, x2, 1.0, f"x2_{i}", toy_title_fwd) for i, x2 in enumerate(TOY_X2_RANGE)]

toy_fwd_init = toy_x1_frames[20]  # x1=0, x2=0
toy_fwd_fig = go.Figure(data=list(toy_fwd_init.data), frames=toy_x1_frames + toy_x2_frames)
toy_fwd_fig.update_layout(
    **_toy_base_layout(),
    annotations=[toy_fwd_init.layout.annotations[0]],
    sliders=[
        dict(y=-0.22, x=0.05, len=0.9, currentvalue=dict(prefix="x1 = "), active=20,
             steps=[dict(method="animate", args=[[f.name], dict(mode="immediate", frame=dict(duration=0, redraw=True), transition=dict(duration=0))],
                          label=f"{x:.1f}") for x, f in zip(TOY_X1_RANGE, toy_x1_frames)]),
        dict(y=-0.50, x=0.05, len=0.9, currentvalue=dict(prefix="x2 = "), active=20,
             steps=[dict(method="animate", args=[[f.name], dict(mode="immediate", frame=dict(duration=0, redraw=True), transition=dict(duration=0))],
                          label=f"{x:.1f}") for x, f in zip(TOY_X2_RANGE, toy_x2_frames)]),
    ],
    height=650, margin=dict(t=110, b=200),
)

HTML(toy_fwd_fig.to_html(include_plotlyjs=True, full_html=False))

#### Plot 2: backward pass

Here, the inputs stay fixed at `x1 = x2 = 0`, the starting point of Plot 1. The slider now sets `v`, which says how much we care about the output. It plays the same role as in the AIFS calculation, where `v` selects the 2 m temperature near 40°N, 120°E. `v` does not change the network or its prediction, it just multiplies the sensitivities `dy/dx`, i.e. we are looking at `v dy/dx`.

You can see this by moving the `v` slider below. Every edge and bar then scales: `v = 2` doubles all the sensitivities, and `v = −1` flips their sign. So the size of `v` only sets the scale of the sensitivities. We will usually just use `v = 1` (a "unit perturbation"), where we care, and `v = 0` where we don't care about the sensitivities. The bars are labelled `v.dy/dx1` and `v.dy/dx2` because they show the gradient multiplied by `v`.

In [ ]:
toy_title_bwd = lambda x1, x2, v, y, d1, d2: (
    f"BACKWARD PASS -- fixed at x1={x1:.2f}, x2={x2:.2f} (same point Plot 1 started from)"
    f"<br>output perturbation v={v:.2f}  =>  backward pass reveals: v.dy/dx1={d1:.2f}   v.dy/dx2={d2:.2f}"
)

TOY_V_RANGE = np.linspace(-2, 2, 41)
toy_v_frames = [_toy_frame(0.0, 0.0, v, f"v_{i}", toy_title_bwd, show_halo=True, bar_labels=("v.dy/dx1", "v.dy/dx2")) for i, v in enumerate(TOY_V_RANGE)]

toy_bwd_init = toy_v_frames[30]  # v=1, the AIFS-default unit perturbation
toy_bwd_fig = go.Figure(data=list(toy_bwd_init.data), frames=toy_v_frames)
toy_bwd_fig.update_layout(
    **_toy_base_layout(),
    annotations=[toy_bwd_init.layout.annotations[0]],
    sliders=[
        dict(y=-0.22, x=0.05, len=0.9, currentvalue=dict(prefix="output perturbation v = "), active=30,
             steps=[dict(method="animate", args=[[f.name], dict(mode="immediate", frame=dict(duration=0, redraw=True), transition=dict(duration=0))],
                          label=f"{x:.1f}") for x, f in zip(TOY_V_RANGE, toy_v_frames)]),
    ],
    height=650, margin=dict(t=110, b=140),
)

HTML(toy_bwd_fig.to_html(include_plotlyjs=True, full_html=False))

This is exactly what we do with AIFS below, with millions of inputs instead of two: we choose `v` at the output, propagate it backwards through the network, and read off how sensitive the output is to every input.

## 1. Where are the sensitivities largest? An overview

This figure shows, for every pressure-level variable (`q` specific humidity, `t` temperature, `u` and `v` wind, `w` vertical velocity, `z` geopotential) and every pressure level, the most negative sensitivity (left, red) and the most positive sensitivity (right, blue) over all grid points. The top row is for the input at −6H, the bottom row for the input at 0H.

In [ ]:
show_artifact("01_pressure_level_summary.png", title="Min/max backward sensitivity by pressure-level variable and level")

Two things stand out. First, the sensitivities to the 0H input are much larger than those to the −6H input: for a 6-hour forecast, the current-state input matters most. Second, the largest values are for geopotential and temperature in the lower half of the atmosphere (below about 400 hPa). That is, 2m temperature at our chosen location and time is most sensitive to to these values. The map, profile and animation below look at these more closely.

## 2. Explore it yourself: an interactive sensitivity map

This map shows the sensitivity to the 0H input at every grid point. You can **pan, zoom, and use the dropdown** to switch between input variables. It starts zoomed in on the region around 40°N, 120°E (the black star). Red means that increasing the input there would raise the forecast 2 m temperature near the star; blue means it would lower it.

- **`2t`** (the default) shows how the forecast 2 m temperature depends on the *initial* 2 m temperature. As you would expect, the sensitivity is positive, and almost all of it (over 90%) lies within 500 km of the star: a warmer start near the star gives a warmer forecast there.
- **`z_500`** shows the sensitivity to geopotential at 500 hPa. Its pattern is different: a strong negative signal right at the star, and a weaker positive one to the north-west. Whether such a pattern makes physical sense is something the sensitivity cannot tell you. That needs a meteorologist's judgement.
- **`t_850`** (temperature at 850 hPa) and **`q_700`** (specific humidity at 700 hPa) are included so you can explore other variables and levels.

In [ ]:
import plotly.graph_objects as go
import cmocean

MAP_VARIABLES = ["2t", "z_500", "t_850", "q_700"]


def cmocean_to_plotly(cmap, n=32):
    return [[i / (n - 1), f"rgb({int(r*255)},{int(g*255)},{int(b*255)})"] for i, (r, g, b, _a) in enumerate(cmap(np.linspace(0, 1, n)))]


colorscale = cmocean_to_plotly(cmocean.cm.balance)

fig = go.Figure()
buttons = []
for i, var in enumerate(MAP_VARIABLES):
    field = sensitivities[var][1]  # 0H timestep
    lim = float(np.abs(field).max())
    fig.add_trace(
        go.Scattergeo(
            lat=lat, lon=lon_wrapped, mode="markers",
            marker=dict(size=5, color=field, colorscale=colorscale, cmin=-lim, cmax=lim, colorbar=dict(title=var)),
            visible=(i == 0), name=var,
        )
    )
    vis = [False] * len(MAP_VARIABLES)
    vis[i] = True
    buttons.append(dict(label=var, method="update", args=[{"visible": vis + [True]}, {"title": f"Backward sensitivity: {var} (0H)"}]))

fig.add_trace(
    go.Scattergeo(
        lat=[PERTURB_LOCATION[0]], lon=[PERTURB_LOCATION[1]], mode="markers",
        marker=dict(size=12, color="black", symbol="star", line=dict(width=1, color="white")),
        name="perturbation point",
    )
)

fig.update_layout(
    title=dict(text="Backward sensitivity: 2t (0H)", x=0.02, xanchor="left"),
    updatemenus=[dict(buttons=buttons, direction="down", x=1.0, y=1.15, xanchor="right", showactive=True)],
    annotations=[
        dict(text="variable:", x=0.87, y=1.15, xref="paper", yref="paper", showarrow=False, xanchor="right"),
        dict(text="scroll to zoom out / drag to pan -- shown region is East Asia by default",
             x=0.02, y=-0.06, xref="paper", yref="paper", showarrow=False, xanchor="left",
             font=dict(size=11, color="grey")),
    ],
    legend=dict(x=0.02, y=0.02, bgcolor="rgba(255,255,255,0.7)"),
    geo=dict(projection_type="natural earth", showcoastlines=True, showcountries=True,
             coastlinecolor="grey", countrycolor="lightgrey",
             lataxis=dict(range=[15, 65]), lonaxis=dict(range=[95, 145])),
    height=600, margin=dict(t=90, b=60),
)

# Embed the map as self-contained HTML (with plotly.js included) rather than using fig.show(),
# so that it also displays in static previews such as GitHub, and without internet access.
from IPython.display import HTML

HTML(fig.to_html(include_plotlyjs=True, full_html=False))

## 3. The vertical structure: a cross-section

The map shows one pressure level at a time. This cross-section instead takes a slice along 40.5°N, through the star, and shows the sensitivity to temperature at all pressure levels (vertical axis) and longitudes (horizontal axis), for the −6H input (left) and the 0H input (right). Note that the titles inside the image say `lon=40.5°`, but the slice is along *latitude* 40.5°N. Also note that in this figure, blue is positive and red is negative: the opposite of the map above.

At 0H, the sensitivity is positive and concentrated between about 115°E and 123°E, below about 700 hPa. So the forecast 2 m temperature mostly depends on the temperature of the lowest few kilometres of the atmosphere directly around it. At −6H, the signal is weaker and has mixed signs.

In [ ]:
show_artifact("05_cross_section_temperature.png", title="Backward sensitivity of the 2t perturbation to initial temperature, cross-section at 40.5N")

## 4. The vertical profile at the star

This plot narrows things down further, to the single grid point closest to 40°N, 120°E. For each variable, it shows how the sensitivity varies with pressure level. Each line is divided by its own largest value, because the raw values differ by an order of magnitude between variables (e.g. 2 m temperature is more sensitive to the temperature profile than to the humidity profile). So the plot shows *where* each variable's sensitivity peaks.

In [ ]:
closest_idx = int(np.argmin(haversine_km(lat, lon, PERTURB_LOCATION[0], PERTURB_LOCATION[1])))
profile_vars = ["t", "q", "z", "u", "v"]

fig, axs = plt.subplots(1, 2, figsize=(11, 6), sharey=True)
for ax, tstep, label in zip(axs, [0, 1], TIMESTEP_LABELS):
    for var in profile_vars:
        values = np.array([sensitivities[f"{var}_{lvl}"][tstep, closest_idx] for lvl in LEVELS])
        norm = values / (np.abs(values).max() + 1e-12)
        ax.plot(norm, LEVELS, marker="o", markersize=4, label=var)
    ax.set_title(f"At {label}")
    ax.set_xlabel("Sensitivity (normalised to each\nvariable's own peak)")
    ax.axvline(0, color="grey", linewidth=0.8)
    ax.set_ylim(1000, 50)

axs[0].set_ylabel("Pressure level (hPa)")
axs[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left")
fig.suptitle(f"Vertical profile at the perturbation point ({PERTURB_LOCATION[0]:.0f}N, {PERTURB_LOCATION[1]:.0f}E)")
plt.tight_layout()
plt.show()

This is quite a noisy graph. But through the noise, you can see that the sensitivity to every variable is largest at 500 hPa or below, i.e. in the lower half of the atmosphere, close to the surface where the 2 m temperature is.

## 5. All 13 pressure levels: an animation

Finally, this animation shows the sensitivity to geopotential (`z`) over the whole region, one pressure level at a time: from 50 hPa (the highest level in the model input) down to 1000 hPa (close to the surface). All frames use the same colour scale, so you can compare them directly. You will see that the sensitivity is small in the upper atmosphere, and largest at 700 and 850 hPa.

In [ ]:
import matplotlib.animation as animation

region = (lat > 15) & (lat < 65) & (lon_wrapped > 95) & (lon_wrapped < 145)
lat_r, lon_r = lat[region], lon_wrapped[region]
fields_by_level = {lvl: sensitivities[f"z_{lvl}"][1, region] for lvl in LEVELS}
vmax = max(np.abs(f).max() for f in fields_by_level.values())

fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(lon_r, lat_r, c=fields_by_level[LEVELS[0]], cmap=cmocean.cm.balance, vmin=-vmax, vmax=vmax, s=18)
ax.scatter(*PERTURB_LOCATION[::-1], marker="*", s=200, c="black", edgecolors="white", zorder=5)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(alpha=0.3)
anim_title = ax.set_title(f"z sensitivity at {LEVELS[0]} hPa (0H)")
plt.colorbar(sc, ax=ax, label="z sensitivity")


def _update(i):
    lvl = LEVELS[i]
    sc.set_array(fields_by_level[lvl])
    anim_title.set_text(f"z sensitivity at {lvl} hPa (0H)")
    return sc, anim_title


anim = animation.FuncAnimation(fig, _update, frames=len(LEVELS), interval=600, blit=False)
plt.close(fig)  # prevent a static duplicate of the last frame from also rendering below
from IPython.display import HTML

HTML(anim.to_jshtml())

---
## Recap

In Part 2, three of the four methods (permutation importance, PDP/ICE and SHAP) were *perturbation-based* and *model-agnostic*: they change the inputs and watch how the output responds, without looking inside the model. Tree importance was the exception: it is read off the fitted model's internal structure.

Backward sensitivities are different again. They are *gradient-based*: we compute them by differentiating *through* the model, so we need access to its internals. In return, a single backward pass gives the sensitivity of one output to every one of the millions of inputs, which would be far too expensive with perturbation-based methods.

**Where it misleads you.**

- **It is only valid for small changes.** A gradient tells you how the output responds to a *small* change in an input, at this particular initial state. For larger changes, or in another weather situation, the response can be very different, as the saturating `tanh` units in the toy network showed.
- **Units matter.** The ranking of variables depends on the units the sensitivities are expressed in; here, the model's normalised units.
- **It explains one forecast.** We explained one output (the 2 m temperature near 40°N, 120°E), at one lead time, from one initial state. That says nothing directly about how AIFS behaves elsewhere or on other days.
- **It explains the model, not the world.** Just like SHAP, a sensitivity describes what AIFS has learned. Whether that matches real atmospheric physics is a separate question.

If you have worked in numerical weather prediction, this method may look familiar. Adjoint models, used e.g. in 4D-Var data assimilation and in forecast sensitivity studies, compute the same kind of gradient: how a forecast quantity depends on the initial state. For a physics-based model, the adjoint has to be derived from the model equations and coded by hand. For a neural network like AIFS, automatic differentiation gives it to you directly.

This is the main message of this module: the domain knowledge you may already have, about which features should matter or how a forecast depends on its initial state, is what you need to judge whether an AI model's explanations make sense.